In [3]:
from sqlalchemy import create_engine
import pandas as pd

# Create SQLite database
engine = create_engine('sqlite:///../artifacts/pulsemap.db')

# Load final dataset
df = pd.read_excel('../artifacts/pulsemap_final.xlsx')

# Table 1 — districts
districts = df[['district', 'state']].copy()
districts.index += 1  # id starts from 1
districts.index.name = 'id'
districts.to_sql('districts', engine, if_exists='replace', index=True)
print("districts table loaded:", len(districts), "rows")

districts table loaded: 707 rows


In [4]:
# Table 2 — health_metrics
health_cols = ['district', 'bmi_below_normal_w', 'overweight_obese_w',
               'anaemia_w', 'blood_sugar_w', 'blood_sugar_m',
               'bp_elevated_w', 'bp_elevated_m', 'tobacco_w',
               'tobacco_m', 'alcohol_w', 'alcohol_m',
               'cervical_screening_w', 'breast_exam_w', 'oral_exam_w']

health_metrics = df[health_cols].copy()
health_metrics['district_id'] = health_metrics.index + 1
health_metrics = health_metrics.drop(columns=['district'])
health_metrics.to_sql('health_metrics', engine, 
                       if_exists='replace', index=False)
print("health_metrics table loaded:", len(health_metrics), "rows")

health_metrics table loaded: 707 rows


In [5]:
# Table 3 — socioeconomic
socio_cols = ['district', 'literacy_w', 'health_insurance',
              'clean_water', 'sanitation', 'clean_fuel']

socioeconomic = df[socio_cols].copy()
socioeconomic['district_id'] = socioeconomic.index + 1
socioeconomic = socioeconomic.drop(columns=['district'])
socioeconomic.to_sql('socioeconomic', engine,
                      if_exists='replace', index=False)
print("socioeconomic table loaded:", len(socioeconomic), "rows")

socioeconomic table loaded: 707 rows


In [6]:
# Table 4 — risk_scores
risk_cols = ['district', 'risk_score', 'risk_tier', 'cluster']
risk_scores = df[risk_cols].copy()
risk_scores['district_id'] = risk_scores.index + 1
risk_scores = risk_scores.drop(columns=['district'])
risk_scores.to_sql('risk_scores', engine,
                    if_exists='replace', index=False)
print("risk_scores table loaded:", len(risk_scores), "rows")

risk_scores table loaded: 707 rows


In [7]:
print("\nAll tables loaded successfully")
print("Database saved at: artifacts/pulsemap.db")


All tables loaded successfully
Database saved at: artifacts/pulsemap.db


In [9]:
from src.logger.logger import logging

In [11]:
# Query 1 — Top 10 Critical Risk Districts
q1 = pd.read_sql("""
    SELECT 
        d.district,
        d.state,
        r.risk_score,
        r.risk_tier
    FROM districts d
    JOIN risk_scores r ON d.id = r.district_id
    WHERE r.risk_tier = 'Critical Risk'
    ORDER BY r.risk_score DESC
    LIMIT 10
""", engine)
logging.info("Query 1 executed — Top 10 Critical Risk Districts")
print("=== Top 10 Critical Risk Districts ===")
print(q1.to_string(index=False))

=== Top 10 Critical Risk Districts ===
           district                     state  risk_score     risk_tier
            Bijapur              Chhattisgarh      100.00 Critical Risk
              Sukma              Chhattisgarh       89.86 Critical Risk
         Mayurbhanj                    Odisha       89.72 Critical Risk
Pashchimi Singhbhum                 Jharkhand       88.46 Critical Risk
              Anjaw         Arunachal Pradesh       87.93 Critical Risk
         Malkangiri                    Odisha       87.30 Critical Risk
          Kendujhar                    Odisha       82.79 Critical Risk
             Bastar              Chhattisgarh       81.33 Critical Risk
             Namsai         Arunachal Pradesh       81.03 Critical Risk
           Nicobars Andaman & Nicobar Islands       80.96 Critical Risk


In [12]:
# Query 2 — State-wise Risk Distribution
q2 = pd.read_sql("""
    SELECT
        d.state,
        COUNT(CASE WHEN r.risk_tier = 'Critical Risk' THEN 1 END) AS critical,
        COUNT(CASE WHEN r.risk_tier = 'High Risk' THEN 1 END) AS high,
        COUNT(CASE WHEN r.risk_tier = 'Moderate Risk' THEN 1 END) AS moderate,
        COUNT(CASE WHEN r.risk_tier = 'Low Risk' THEN 1 END) AS low,
        COUNT(*) AS total_districts,
        ROUND(AVG(r.risk_score), 2) AS avg_risk_score
    FROM districts d
    JOIN risk_scores r ON d.id = r.district_id
    GROUP BY d.state
    ORDER BY avg_risk_score DESC
    LIMIT 10
""", engine)
logging.info("Query 2 executed — State-wise Risk Distribution")
print("=== Top 10 States by Avg Risk Score ===")
print(q2.to_string(index=False))

=== Top 10 States by Avg Risk Score ===
                    state  critical  high  moderate  low  total_districts  avg_risk_score
                  Tripura         8     0         0    0                8           71.42
                   Odisha        15    14         1    0               30           61.53
Andaman & Nicobar Islands         2     0         1    0                3           61.45
                Jharkhand        10    13         1    0               24           59.34
             Chhattisgarh        11     9         5    2               27           56.04
                   Sikkim         1     3         0    0                4           55.26
        Arunachal Pradesh         6     9         4    1               20           53.90
                    Assam        13    14         6    0               33           53.61
              West Bengal         3    15         2    0               20           51.92
                  Manipur         0     9         0    0    

In [13]:
# Query 3 — Top 3 Districts per State (CTE)
q3 = pd.read_sql("""
    WITH ranked_districts AS (
        SELECT
            d.district,
            d.state,
            r.risk_score,
            r.risk_tier,
            ROW_NUMBER() OVER (
                PARTITION BY d.state
                ORDER BY r.risk_score DESC
            ) AS rn
        FROM districts d
        JOIN risk_scores r ON d.id = r.district_id
    )
    SELECT district, state, risk_score, risk_tier
    FROM ranked_districts
    WHERE rn <= 3
    ORDER BY state, risk_score DESC
""", engine)
logging.info("Query 3 executed — Top 3 districts per state")
print("=== Top 3 Highest Risk Districts Per State ===")
print(q3.to_string(index=False))

=== Top 3 Highest Risk Districts Per State ===
              district                                  state  risk_score     risk_tier
              Nicobars              Andaman & Nicobar Islands       80.96 Critical Risk
North & Middle Andaman              Andaman & Nicobar Islands       63.89 Critical Risk
         South Andaman              Andaman & Nicobar Islands       39.51 Moderate Risk
              Prakasam                         Andhra Pradesh       40.21 Moderate Risk
         East Godavari                         Andhra Pradesh       39.75 Moderate Risk
         Visakhapatnam                         Andhra Pradesh       37.56 Moderate Risk
                 Anjaw                      Arunachal Pradesh       87.93 Critical Risk
                Namsai                      Arunachal Pradesh       81.03 Critical Risk
         Dibang Valley                      Arunachal Pradesh       76.63 Critical Risk
            Hailakandi                                  Assam       73.33

In [14]:
# Query 4 — Gender Health Gap
q4 = pd.read_sql("""
    SELECT
        d.state,
        ROUND(AVG(h.blood_sugar_w), 2) AS avg_bs_women,
        ROUND(AVG(h.blood_sugar_m), 2) AS avg_bs_men,
        ROUND(AVG(h.blood_sugar_m) - AVG(h.blood_sugar_w), 2) AS bs_gap,
        ROUND(AVG(h.bp_elevated_w), 2) AS avg_bp_women,
        ROUND(AVG(h.bp_elevated_m), 2) AS avg_bp_men,
        ROUND(AVG(h.bp_elevated_m) - AVG(h.bp_elevated_w), 2) AS bp_gap
    FROM districts d
    JOIN health_metrics h ON d.id = h.district_id
    GROUP BY d.state
    ORDER BY bp_gap DESC
    LIMIT 10
""", engine)
logging.info("Query 4 executed — Gender health gap")
print("=== Top 10 States — Gender Health Gap ===")
print(q4.to_string(index=False))

=== Top 10 States — Gender Health Gap ===
                    state  avg_bs_women  avg_bs_men  bs_gap  avg_bp_women  avg_bp_men  bp_gap
              Uttarakhand          9.71       13.20    3.50         21.80       31.53    9.72
             Nct Of Delhi         12.11       14.06    1.95         24.19       32.96    8.78
                  Manipur         12.54       14.86    2.32         21.84       30.56    8.72
        Arunachal Pradesh          8.31       11.64    3.34         24.97       33.42    8.45
                   Sikkim         11.53       15.74    4.21         36.07       44.14    8.08
                 Nagaland          9.16       12.79    3.63         22.66       30.10    7.43
Andaman & Nicobar Islands         16.06       17.26    1.20         28.57       35.36    6.79
                  Mizoram         13.53       14.89    1.36         16.31       23.08    6.77
                   Punjab         14.83       14.22   -0.61         30.87       37.26    6.39
               Cha

## SQL Analysis — Key Findings

### Query 1: Top 10 Critical Risk Districts
Bijapur (Chhattisgarh) scores 100/100 — maximum risk district in India.
Chhattisgarh and Odisha dominate the top 10, with tribal districts
showing extreme multi-dimensional health deprivation.

### Query 2: State-wise Risk Distribution
| State | Critical | High | Moderate | Low | Avg Score |
|---|---|---|---|---|---|
| Tripura | 8 | 0 | 0 | 0 | 71.42 |
| Odisha | 15 | 14 | 1 | 0 | 61.53 |
| Andaman & Nicobar | 2 | 0 | 1 | 0 | 61.45 |
| Jharkhand | 10 | 13 | 1 | 0 | 59.34 |
| Chhattisgarh | 11 | 9 | 5 | 2 | 56.04 |

**Critical insight:** Tripura has 100% of its districts in Critical 
Risk tier every single district needs immediate intervention.
This is not a district problem, it is a complete state-level 
health crisis.

### Query 3: Top 3 Districts Per State (CTE + Window Function)
Every state has at least one High or Critical Risk district 
no state in India is completely free from preventable disease burden.
Tribal districts consistently appear as top risk across states 
Bijapur, Sukma, Bastar (Chhattisgarh), Anjaw, Namsai (Arunachal).

### Query 4: Gender Health Gap
- **Uttarakhand** has highest BP gap men 9.72% higher than women.
- **Punjab and Chandigarh** show negative blood sugar gap 
  women have higher blood sugar than men, unique pattern possibly 
  linked to dietary habits and lower physical activity in women.
- **Sikkim** shows highest blood sugar gap (4.21) men significantly 
  more diabetic than women.

**Overall:** Men consistently show higher BP across all states 
cardiovascular risk in men is a national pattern, not regional.